# From Belief to Code: A Comprehensive Tutorial on Bayesian Networks with Python and pgmpy

## Part 1: The Bayesian Paradigm: A New Way of Reasoning About Uncertainty

At the heart of modern data science, machine learning, and artificial intelligence lies the challenge of reasoning under uncertainty. For over a century, the field of statistics has been dominated by a particular school of thought. Yet, a powerful and increasingly influential alternative offers not just a different set of tools, but a fundamentally different philosophy for quantifying what we know and how we learn. This is the Bayesian paradigm, a framework that treats probability as a degree of belief, which can be systematically updated in the light of new evidence. This tutorial provides a comprehensive guide to this paradigm, moving from its foundational principles to its practical implementation in building complex probabilistic models with the `pgmpy` library in Python.

### 1.1 Two Schools of Thought: A Tale of Two Probabilities

The discipline of statistics is marked by a deep philosophical divide between two primary approaches: the Frequentist and the Bayesian.[1] Understanding this distinction is the first step toward appreciating the unique power of Bayesian methods.

**The Frequentist View**

The Frequentist approach, which forms the basis of most introductory statistics courses, defines probability as the long-term frequency of an event over many hypothetical repetitions of an experiment. Consider the canonical example of flipping a fair coin. A Frequentist would assert that the probability of getting heads is 0.5. This assertion is not a statement about a single coin flip; rather, it's a claim that if one were to flip the coin an infinite number of times, the proportion of heads would converge to 50%.[1, 2]

In this framework, the parameters of a population—such as the true mean of a distribution or the true coefficient in a regression model—are considered fixed, unknown constants.[3, 4] The goal of a statistical analysis is to use observed data, which is treated as a random sample, to estimate these fixed constants. The uncertainty captured by concepts like a "95% confidence interval" is not uncertainty about the parameter itself. Instead, it is uncertainty about the estimation procedure; it means that if the experiment were repeated many times, 95% of the calculated intervals would contain the true, fixed parameter. This interpretation is often counterintuitive and frequently misunderstood.[5]

**The Bayesian View**

The Bayesian approach offers a more intuitive definition of probability: it is a measure of subjective belief, confidence, or the state of knowledge about a proposition.[3, 6] This perspective allows one to assign probabilities to events that cannot be repeated, such as the probability that a specific candidate will win an election or that the Chicago Bulls would win the championship in 2001—questions for which the Frequentist must remain silent.[7]

Crucially, in the Bayesian world, a parameter is not a fixed constant but a random variable about which we can have a distribution of beliefs.[1, 4] Our analysis begins with a **prior probability distribution**, which encodes our initial beliefs about the parameter before observing any data. As we collect data, we use a mathematical engine called Bayes' Theorem to update our initial beliefs, resulting in a **posterior probability distribution**. This posterior distribution represents our refined state of knowledge, combining our prior beliefs with the information provided by the evidence.[2, 8, 9]

This philosophical difference leads to profoundly different outputs. A Frequentist analysis might yield a p-value to help decide whether to reject a null hypothesis (e.g., "is there a statistically significant difference between A and B?").[2, 8] A Bayesian analysis, in contrast, provides a full probability distribution for the parameter of interest, directly answering questions like, "What is the probability that A is better than B, and by how much?".[8, 10] The Bayesian output, such as a 95% credible interval, carries a more direct interpretation: there is a 95% probability that the true value of the parameter lies within this range, given the data and the model.[5] This directness and interpretability are major reasons for the growing adoption of Bayesian methods in complex fields.

| Aspect | Frequentist Approach | Bayesian Approach |
| :--- | :--- | :--- |
| **Philosophy of Probability** | Objective long-run frequency of events.[2, 11] | Subjective degree of belief or confidence.[3, 6] |
| **Treatment of Parameters** | Fixed, unknown constants.[4, 9] | Random variables with probability distributions.[1, 4] |
| **Core Method** | Hypothesis testing (p-values), Maximum Likelihood Estimation.[2, 8] | Bayes' Theorem to update beliefs (Priors -> Posteriors).[8, 9] |
| **Primary Output** | Point estimates, confidence intervals, p-values.[2, 9] | Full posterior probability distributions, credible intervals.[10, 12] |
| **Role of Prior Knowledge** | Not formally incorporated; objectivity is key.[11] | Formally incorporated via a "prior" distribution.[2, 12] |

### 1.2 The Engine of Belief-Updating: Bayes' Theorem

The mathematical cornerstone of Bayesian statistics is Bayes' Theorem. While its derivation is a straightforward consequence of the definition of conditional probability, its implications are profound.[13, 14] It provides the formal mechanism for learning from experience and updating our beliefs.

The theorem is stated as:

$$P(H|E) = \frac{P(E|H) P(H)}{P(E)}$$

Where:
* $P(H|E)$ is the **posterior probability**: the probability of our hypothesis $H$ being true, *given* the evidence $E$. This is our updated belief.[9, 15]
* $$P(H)$$ is the **prior probability**: our initial belief in the hypothesis $H$ *before* considering the new evidence.[16]
* $P(E|H)$ is the **likelihood**: the probability of observing the evidence $E$ *if* our hypothesis $H$ were true.[16, 17]
* $P(E)$ is the **evidence** (or marginal likelihood): the total probability of observing the evidence $E$ under all possible hypotheses. It acts as a normalization constant.[13, 18]

To make this tangible, consider a classic medical diagnosis problem, which illustrates how human intuition often fails where Bayesian reasoning succeeds.[18, 19]

Suppose 1% of women in a certain demographic have a particular type of cancer. A mammogram test is developed; it correctly identifies 80% of women who have the cancer (true positive rate) but also reports a positive result for 9.6% of women who do not have cancer (false positive rate). If a woman from this demographic receives a positive mammogram, what is the probability she actually has cancer?

Many people intuitively jump to a high number, near 80%. However, Bayes' Theorem reveals a very different and counterintuitive answer.

* **Hypothesis (H):** The woman has cancer.
* **Evidence (E):** The woman has a positive mammogram.

Let's break down the components:
* **Prior $P(H)$:** The prevalence of cancer is 1%, so $P(H) = 0.01$. This is our starting belief.
* **Likelihood $P(E|H)$:** The probability of a positive test given cancer (the true positive rate) is 80%, so $P(E|H) = 0.80$.
* **Evidence $P(E)$:** This is the total probability of getting a positive test. It's the sum of true positives and false positives.
    * Chance of a true positive = $P(E|H) \times P(H) = 0.80 \times 0.01 = 0.008$.
    * Chance of a false positive = $P(E|\neg H) \times P(\neg H) = 0.096 \times (1 - 0.01) = 0.096 \times 0.99 \approx 0.095$.
    * Therefore, $P(E) = 0.008 + 0.095 = 0.103$. About 10.3% of all women will test positive.
* **Posterior $P(H|E)$:** Now we can calculate our final answer.
    $P(H|E) = (0.80 \times 0.01) / 0.103 = 0.008 / 0.103 \approx 0.078$.

The updated probability is only 7.8%. A positive test result doesn't make cancer an 80% certainty; it just "slides" the initial 1% probability up to 7.8%.[19] The reason is that false positives from the large population of healthy women (99%) are far more numerous than true positives from the small population of sick women (1%).

A more intuitive way to think about the theorem is to see it as adjusting a prior belief by an "update factor" [13]:

$P(H|E) = P(H) \times \frac{P(E|H)}{P(E)}$

The term $P(E|H) / P(E)$ captures the strength of the evidence. It measures how much more likely the evidence is if our hypothesis is true, compared to its general likelihood. If this ratio is large, it means the evidence is a strong indicator, and our belief will be updated significantly.[13, 14]

| Component | Formula Term | Intuitive Question | Example (from [18, 19]) |
| :--- | :--- | :--- | :--- |
| **Prior** | `$P(H)$` | How common is the condition? | "What is the probability a woman has cancer?" (1%) |
| **Likelihood** | `$P(E|H)$` | How good is the test at finding the condition? | "If she has cancer, what's the chance of a positive test?" (80%) |
| **Evidence** | `$P(E)$` | How common is a positive test overall? | "What's the total chance of a positive test (true or false)?" (10.3%) |
| **Posterior** | `$P(H|E)$` | Given the test result, how likely is the condition? | "If her test is positive, what's the chance she has cancer?" (7.8%) |

## Part 2: Structuring Knowledge: An Introduction to Bayesian Networks

Bayes' Theorem provides a powerful rule for updating our belief about a single hypothesis based on a single piece of evidence. However, real-world problems involve dozens or even hundreds of interconnected variables. Modeling such systems requires a more scalable framework. This is the role of Bayesian Networks.

### 2.1 Beyond Single Equations: Probabilistic Graphical Models

Imagine trying to model the factors affecting a student's grade. These might include the difficulty of the course, the student's intelligence, their study habits, their SAT score, and whether they get a good recommendation letter. To model this system comprehensively, one might try to construct a full joint probability distribution, $P(\text{Difficulty, Intelligence, Grade, SAT, Letter})$. However, even with just five binary variables, this would require specifying $2^5 - 1 = 31$ probabilities. For a system with 30 such variables, the number of probabilities required would exceed one billion. This "curse of dimensionality" makes building and reasoning with full joint distributions computationally intractable for all but the simplest problems.[20]

Probabilistic Graphical Models (PGMs) are a class of data structures designed to solve this problem. They use a graph to represent a complex probability distribution in a compact way by exploiting the conditional independence relationships between variables.[7, 20, 21] A Bayesian Network (BN) is the most common type of PGM, providing a "skeleton" that allows us to factorize the large, unwieldy joint distribution into a product of smaller, manageable local distributions.[22]

### 2.2 The Anatomy of a Bayesian Network

A Bayesian Network consists of two fundamental components that define the model's qualitative structure and its quantitative parameters.[15, 23, 24]

* **Nodes (Vertices):** Each node in the graph represents a random variable of interest in the system. These variables can be discrete, taking on a finite set of values (e.g., `Smoker` = {Yes, No}), or continuous (e.g., `Temperature`). In a BN, nodes are the fundamental building blocks representing the concepts we want to model.[15, 23]
* **Edges (Arcs):** The nodes are connected by directed arrows, or edges. An edge from a node `A` to a node `B` ($A \rightarrow B$) signifies that `A` has a direct probabilistic influence on `B`. In this relationship, `A` is called the **parent** of `B`, and `B` is the **child** of `A`.[24, 25] These edges encode the conditional dependencies within the model; the state of a child node is probabilistically dependent on the state of its parent nodes.

### 2.3 The Importance of Being Acyclic: Directed Acyclic Graphs (DAGs)

The graphical structure of a Bayesian Network is not just any directed graph; it must be a **Directed Acyclic Graph (DAG)**.[15, 17] This constraint is critical to the logical consistency and computational function of the network.

* **Directed:** As described, the edges have a specific direction, indicating the flow of influence or causality from parent to child.[15]
* **Acyclic:** The graph must not contain any directed cycles. This means it is impossible to start at a node, follow a sequence of directed edges, and arrive back at the starting node.[25, 26] A path like $A \rightarrow B \rightarrow C \rightarrow A$ is forbidden.

The acyclic constraint is essential because a cycle would imply a logical paradox. If $A$ influences $B$, and $B$ influences $A$, then $A$ is effectively its own ancestor. This creates an infinitely recursive definition that is mathematically and conceptually nonsensical.[25, 26] The DAG structure ensures that a clear "topological ordering" of the variables is always possible, which is fundamental to how the network defines the joint probability distribution and how inference algorithms operate.[26]

The true power of a BN lies in how this DAG structure allows for a massive simplification of the joint probability distribution. The chain rule of probability allows any joint distribution to be expressed as:
$P(X_1, X_2, \dots, X_n) = P(X_1) \times P(X_2|X_1) \times \dots \times P(X_n|X_1, \dots, X_{n-1})$

This decomposition corresponds to a fully connected graph and offers no computational savings. However, the structure of a BN makes a powerful assertion: **a node is conditionally independent of all its non-descendants, given its parents**.[27] This assumption allows us to simplify the chain rule dramatically. The joint probability distribution "factorizes" according to the graph structure into a product of local conditional probabilities for each node given its parents [17, 22, 27]:

$$P(X_1, \dots, X_n) = \prod_{i=1}^{n} P(X_i | \text{Parents}(X_i))$$

For example, in a simple chain structure $A \rightarrow B \rightarrow C$, the joint probability is not $P(A)P(B|A)P(C|A,B)$. Instead, because the graph asserts that $C$ is independent of $A$ *given* $B$, the joint probability simplifies to $P(A)P(B|A)P(C|B)$.[28] This factorization is the key to a BN's efficiency. It replaces the need for one enormous table representing the full joint distribution with a collection of much smaller, local tables—one for each node.

## Part 3: Quantifying Relationships: The Role of Conditional Probability Tables (CPTs)

Once the qualitative structure of the domain is defined by the DAG, the next step is to specify the quantitative relationships between the variables. For Bayesian Networks with discrete variables, this is accomplished using Conditional Probability Tables (CPTs).[29, 30]

### 3.1 What is a Conditional Probability Table?

A CPT is the numerical engine of a discrete Bayesian Network.[24] Every node in the network has a CPT associated with it that quantifies the probability distribution of that node's states, conditional on the states of its parents.[27, 31, 32]

* For a **root node** (a node with no parents), the CPT is not conditional. It is simply a table defining the node's **marginal probability distribution**, often referred to as its prior probability.[23, 24] For a variable `Difficulty` with states {Easy, Hard}, its CPT would specify $P(\text{Difficulty=Easy})$ and $P(\text{Difficulty=Hard})$.
* For a **child node** (a node with one or more parents), the CPT specifies the probability distribution of the child for *every possible combination* of its parents' states. For a node `Grade` with parent `Difficulty`, the CPT would specify $P(\text{Grade} | \text{Difficulty=Easy})$ and $P(\text{Grade} | \text{Difficulty=Hard})$. Each column in a CPT corresponds to a specific configuration of the parent states, and the probabilities within that column must sum to 1.[24, 33]

These tables can be populated from expert knowledge, which is common when empirical data is scarce, or they can be learned directly from data.[7, 31]

### 3.2 Constructing a CPT from Data: A Manual Walkthrough

Before diving into the automated methods in `pgmpy`, it is invaluable to understand how a CPT can be constructed manually from raw data. This process demystifies the concept of "parameter learning" and reveals that the Maximum Likelihood Estimation (MLE) approach used by many libraries is, at its core, a straightforward process of calculating conditional frequencies.[34]

Let's walk through the construction of the CPT for $P(\text{Grade} | \text{Difficulty})$ from a hypothetical dataset of student outcomes.

**Step 1: Obtain the Raw Data**
The process begins with a dataset, typically in a tabular format like a pandas DataFrame, containing observations for the variables of interest.

**Part A: Raw Data Snippet (pandas DataFrame `df`)**

| StudentID | Difficulty | Grade |
| :--- | :--- | :--- |
| 1 | Easy | B |
| 2 | Hard | C |
| 3 | Easy | A |
|... |... |... |
| 100 | Easy | B |

**Step 2: Create a Frequency Table**
The next step is to summarize the data by counting the co-occurrences of each state combination. A two-way table, also known as a contingency table, is perfect for this task.[35, 36] This table shows the raw counts for each joint event (e.g., the number of students who took an 'Easy' course and received a 'B').

**Part B: Frequency Count Table (e.g., `pd.crosstab(df.Grade, df.Difficulty)`)**

| Difficulty | Easy | Hard |
| :--- | :--- | :--- |
| **Grade** | | |
| A | 20 | 5 |
| B | 30 | 15 |
| C | 10 | 20 |
| **Total**| **60** | **40** |

**Step 3: Calculate Conditional Probabilities**
The final step is to convert these raw counts into conditional probabilities. To find $P(\text{Grade} | \text{Difficulty})$, we must normalize the counts within each column (i.e., for each condition of the parent variable `Difficulty`).

* For the condition `Difficulty='Easy'`, there are 60 total students. The probability of getting an 'A' *given* the course was 'Easy' is the count of (Grade='A' AND Difficulty='Easy') divided by the total count of (Difficulty='Easy').
    $P(\text{Grade='A'} | \text{Difficulty='Easy'}) = 20 / 60 \approx 0.33$
* This process is repeated for every cell, dividing each count by its column total.[37, 38]

This procedure yields the final Conditional Probability Table. This table is the direct input required by `pgmpy` to quantify the relationship $Difficulty \rightarrow Grade$. Understanding this manual derivation makes the automated `.fit()` method in `pgmpy` a transparent tool rather than an opaque black box.

**Part C: Final Conditional Probability Table (CPT)**

| '$P(\text{Grade}|(\text{Difficulty})$' | '`Difficulty='Easy'' | Difficulty='Hard'|
| :--- | :--- | :--- |
| **Grade='A'** | 20/60 = 0.333 | 5/40 = 0.125 |
| **Grade='B'** | 30/60 = 0.500 | 15/40 = 0.375 |
| **Grade='C'** | 10/60 = 0.167 | 20/40 = 0.500 |

## Part 4: Building Your First Bayesian Network with `pgmpy`

With a solid theoretical foundation in place, it is time to translate these concepts into working Python code. The `pgmpy` library is a powerful and flexible tool for creating, parameterizing, and performing inference on probabilistic graphical models.[21] This section provides a step-by-step guide to building a Bayesian Network from scratch, using the well-known "Student" model as a running example.[39, 40]

### 4.1 Setting Up Your Environment

First, the `pgmpy` library needs to be installed. It can be installed using either `pip` or `conda`. You can run the following command in your terminal or in a code cell by prepending it with `!`. 

In [3]:
#This is commented out as this normally runs in an environment where I have pre-installed the python modules needed. 
#Uncomment below if you need to load pandas and pygmpy
#Using pip
#!pip install pgmpy pandas

Once installed, the necessary classes can be imported into a Python script. For building a discrete Bayesian Network, the two essential classes are `BayesianNetwork` and `TabularCPD`.

In [4]:
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

### 4.2 Defining the Model Structure (The DAG)

The first step in creating a model is to define its graphical structure—the Directed Acyclic Graph. In `pgmpy`, this is done by creating an instance of the `BayesianNetwork` class and passing it a list of tuples. Each tuple `(parent, child)` represents a directed edge in the graph.[39, 40, 42]

For the Student model, the relationships are:
* Difficulty (`D`) influences Grade (`G`)
* Intelligence (`I`) influences Grade (`G`)
* Intelligence (`I`) influences SAT score (`S`)
* Grade (`G`) influences the quality of a recommendation Letter (`L`)

This structure is defined in `pgmpy` as follows:

In [5]:
# Defining the network structure
student_model = DiscreteBayesianNetwork([
    ('D', 'G'),
    ('I', 'G'),
    ('I', 'S'),
    ('G', 'L')
])

### 4.3 Implementing CPTs in Code with `TabularCPD`

With the skeleton of the model in place, the next step is to quantify the probabilistic relationships using `TabularCPD`. A `TabularCPD` object must be created for each node in the graph.

A frequent point of confusion for new users is the structure of the `values` parameter, especially for nodes with multiple parents. The key rule is that the columns of the `values` array correspond to a lexicographical ordering of the parent states, as if generated by nested loops.[33, 39]

Let's define the CPTs for the Student model, assuming binary states for `D`, `I`, `S`, `L` and three states for `G`.

**1. CPT for Difficulty (`D`) - A Root Node**
`Difficulty` has no parents, so its CPT is a simple prior probability.
States: `D` = {Easy, Hard}

In [6]:
# CPT for Difficulty
cpd_d = TabularCPD(variable='D', variable_card=2,
                   values=[[0.6], [0.4]],
                   state_names={'D': ['Easy', 'Hard']})

**2. CPT for Intelligence (`I`) - A Root Node**
`Intelligence` also has no parents.
States: `I` = {Dumb, Intelligent}

In [7]:
# CPT for Intelligence
cpd_i = TabularCPD(variable='I', variable_card=2,
                   values=[[0.7], [0.3]],
                   state_names={'I': ['Dumb', 'Intelligent']})

**3. CPT for SAT Score (`S`) - A Child Node**
`SAT` has one parent: `Intelligence`.
States: `S` = {Low, High}

In [8]:
# CPT for SAT Score
cpd_s = TabularCPD(variable='S', variable_card=2,
                   values=[[0.95, 0.2],
                           [0.05, 0.8]],
                   evidence=['I'],
                   evidence_card=[2],
                   state_names={'S': ['Low', 'High'],
                                'I': ['Dumb', 'Intelligent']})

**4. CPT for Grade (`G`) - A Child Node with Multiple Parents**
`Grade` has two parents: `Difficulty` and `Intelligence`. This is the most complex CPT.
States: `G` = {A, B, C}
The `evidence` list is `['I', 'D']`. The column ordering will be (I=Dumb, D=Easy), (I=Dumb, D=Hard), (I=Intelligent, D=Easy), (I=Intelligent, D=Hard).

| $P(G | I, D)$ | I=Dumb, D=Easy | I=Dumb, D=Hard | I=Intelligent, D=Easy | I=Intelligent, D=Hard |
| :--- | :--- | :--- | :--- | :--- |
| **G='A'** | 0.30 | 0.05 | 0.90 | 0.50 |
| **G='B'** | 0.40 | 0.25 | 0.08 | 0.30 |
| **G='C'** | 0.30 | 0.70 | 0.02 | 0.20 |

In [9]:
# CPT for Grade
cpd_g = TabularCPD(variable='G', variable_card=3,
                   values=[[0.3, 0.05, 0.9, 0.5],
                           [0.4, 0.25, 0.08, 0.3],
                           [0.3, 0.7, 0.02, 0.2]],
                   evidence=['I', 'D'],
                   evidence_card=[2, 2],
                   state_names={'G': ['A', 'B', 'C'],
                                'I': ['Dumb', 'Intelligent'],
                                'D': ['Easy', 'Hard']})

**5. CPT for Letter (`L`) - A Child Node**
`Letter` has one parent: `Grade`.
States: `L` = {Bad, Good}

In [10]:
# CPT for Letter
cpd_l = TabularCPD(variable='L', variable_card=2,
                   values=[[0.1, 0.4, 0.99],
                           [0.9, 0.6, 0.01]],
                   evidence=['G'],
                   evidence_card=[3],
                   state_names={'L': ['Bad', 'Good'],
                                'G': ['A', 'B', 'C']})

### 4.4 Assembling and Validating the Model

After defining all the individual CPTs, they must be associated with the model structure using the `add_cpds` method.[39, 43]

In [11]:
# Associating the CPDs with the network
student_model.add_cpds(cpd_d, cpd_i, cpd_s, cpd_g, cpd_l)

The final and most crucial step before proceeding is to validate the model. The `check_model()` method performs several critical checks: it verifies that the graph is a DAG, that every node has an associated CPT, and that the probabilities in each CPT column correctly sum to 1. If the model is consistent, it returns `True`.[39, 43]

In [12]:
# Checking the model for correctness
is_valid = student_model.check_model()
print(f"Is the model valid? {is_valid}")

Is the model valid? True


## Part 5: Activating Your Model: Probabilistic Inference with `pgmpy`

Building a Bayesian Network is only the first half of the journey. The real value comes from using the completed model to answer questions and make predictions—a process known as probabilistic inference. Inference allows us to compute the probability distribution of certain variables of interest (the query) after observing the values of other variables (the evidence).[40]

### 5.1 The Power of Inference: Asking Your Model Questions

`pgmpy` provides a powerful and unified inference engine that is deliberately separated from the model definition. This modular design allows different algorithms to be used with the same model.[44, 45] The two most fundamental types of queries are:

1.  **Probabilistic Query (`query`):** This computes the posterior probability distribution of one or more variables, given some evidence. For example, "What is the probability of a student getting a Grade 'A' given they are Intelligent?".[46]
2.  **Most Probable Explanation (MAP Query) (`map_query`):** This finds the most likely state (or combination of states) for a set of variables, given some evidence. For example, "What is the most likely Grade for an Intelligent student in an Easy course?".[46]

### 5.2 Exact Inference with Variable Elimination

For many networks, exact inference is feasible and provides precise answers. The most common algorithm for this is **Variable Elimination (VE)**, which `pgmpy` implements in the `VariableElimination` class.[44, 46] This algorithm works by systematically eliminating (summing out) all variables that are not part of the query, resulting in the desired marginal or conditional distribution.

**Initialization**
First, the inference engine is initialized by passing the fully specified model to its constructor.

In [13]:
from pgmpy.inference import VariableElimination

# Initialize the inference engine
infer = VariableElimination(student_model)

**Querying for Marginal Probability**
A marginal query calculates the overall probability distribution for a variable before any evidence is observed. This provides a baseline understanding of the model's predictions.

In [14]:
# What is the overall probability distribution of Grade?
grade_dist = infer.query(variables=['G'])
print(grade_dist)

+------+----------+
| G    |   phi(G) |
+======+==========+
| G(A) |   0.3620 |
+------+----------+
| G(B) |   0.2884 |
+------+----------+
| G(C) |   0.3496 |
+------+----------+


**Querying for Conditional Probability (Inference with Evidence)**
This is the most common use case for inference. Evidence is provided as a dictionary where keys are variable names and values are their observed states.

Let's ask: "What is the probability distribution for `Grade` given that the student is `Intelligent` and the course is `Hard`?"

In [15]:
# What is P(Grade | Intelligence='Intelligent', Difficulty='Hard')?
grade_conditional_dist = infer.query(
    variables=['G'],
    evidence={'I': 'Intelligent', 'D': 'Hard'}
)
print(grade_conditional_dist)

+------+----------+
| G    |   phi(G) |
+======+==========+
| G(A) |   0.5000 |
+------+----------+
| G(B) |   0.3000 |
+------+----------+
| G(C) |   0.2000 |
+------+----------+


**Making MAP Queries**
Instead of a full distribution, sometimes the goal is to find the single most likely outcome. The `map_query` method achieves this.

Let's ask: "What is the most likely `Grade` and `Letter` combination for a student who is not intelligent (`Dumb`) but took an `Easy` course?"

In [16]:
# What is the most probable Grade and Letter given I='Dumb', D='Easy'?
map_result = infer.map_query(
    variables=['G', 'L'],
    evidence={'I': 'Dumb', 'D': 'Easy'}
)
print(map_result)

0it [00:00, ?it/s]

0it [00:00, ?it/s]

{'G': 'C', 'L': 'Bad'}


The `pgmpy` inference framework provides a clean and powerful API to unlock the predictive capabilities of a Bayesian Network. By separating the model from the inference algorithm, it allows for flexibility and scalability, enabling users to choose the right tool for their specific problem, from exact methods like Variable Elimination to approximate methods for larger, more complex networks.

## Part 6: A Glimpse into Advanced Topics: Learning Networks from Data

The tutorial so far has focused on building Bayesian Networks when the structure (the DAG) and the parameters (the CPTs) are known beforehand, a common scenario when relying on expert knowledge.[7, 31] However, one of the most powerful features of the Bayesian Network framework—and the `pgmpy` library—is its ability to learn these components directly from data. This capability places BNs at the intersection of knowledge representation and machine learning.

### 6.1 Learning Parameters and Structure

There is a full spectrum of possibilities when it comes to combining domain knowledge and data, and `pgmpy` provides tools for each scenario.[45, 47]

**Parameter Learning**
In many real-world applications, an expert can confidently define the causal structure of a system (the DAG) but lacks the precise numbers to populate the CPTs. In this case, if a dataset is available, `pgmpy` can learn the CPTs automatically. This process, known as parameter learning, automates the manual frequency-counting procedure detailed in Part 3.

The library offers two main estimators:
* **Maximum Likelihood Estimation (`MaximumLikelihoodEstimator`):** This method finds the CPT values that maximize the probability of observing the given data. It is equivalent to using the relative frequencies from the data.[34]
* **Bayesian Estimation (`BayesianEstimator`):** This method starts with a prior belief about the CPTs and updates it using the data. This is useful for avoiding probabilities of zero when certain events don't appear in a small dataset.[34]

The most convenient way to perform parameter learning is with the `fit()` method of a `BayesianNetwork` object.

In [17]:
from pgmpy.estimators import MaximumLikelihoodEstimator

# Create a dummy dataset that reflects the CPTs we defined
# In a real scenario, this would be your observed data
raw_data = {
    'D': ['Easy', 'Easy', 'Hard', 'Easy', 'Hard', 'Hard', 'Easy', 'Easy', 'Hard', 'Easy'],
    'I': ['Dumb', 'Intelligent', 'Dumb', 'Dumb', 'Intelligent', 'Intelligent', 'Dumb', 'Intelligent', 'Intelligent', 'Dumb'],
    'G': ['B', 'A', 'C', 'C', 'B', 'A', 'B', 'A', 'B', 'C'],
    'S': ['Low', 'High', 'Low', 'High', 'High', 'High', 'Low', 'Low', 'High', 'Low'],
    'L': ['Good', 'Good', 'Bad', 'Bad', 'Good', 'Good', 'Bad', 'Good', 'Good', 'Bad']
}
data = pd.DataFrame(raw_data)

# We use the same model structure as before
model_from_data = DiscreteBayesianNetwork([
    ('D', 'G'),
    ('I', 'G'),
    ('I', 'S'),
    ('G', 'L')
])

# Learn the CPTs from data using Maximum Likelihood Estimation
model_from_data.fit(data, estimator=MaximumLikelihoodEstimator)

# The model now has learned CPTs and is ready for inference.
# Let's inspect one of the learned CPTs
print(model_from_data.get_cpds('G'))

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'D': 'C', 'I': 'C', 'G': 'C', 'S': 'C', 'L': 'C'}


+------+---------+----------------+---------+--------------------+
| D    | D(Easy) | D(Easy)        | D(Hard) | D(Hard)            |
+------+---------+----------------+---------+--------------------+
| I    | I(Dumb) | I(Intelligent) | I(Dumb) | I(Intelligent)     |
+------+---------+----------------+---------+--------------------+
| G(A) | 0.0     | 1.0            | 0.0     | 0.3333333333333333 |
+------+---------+----------------+---------+--------------------+
| G(B) | 0.5     | 0.0            | 0.0     | 0.6666666666666666 |
+------+---------+----------------+---------+--------------------+
| G(C) | 0.5     | 0.0            | 1.0     | 0.0                |
+------+---------+----------------+---------+--------------------+


**Structure Learning**
The most data-driven approach involves situations where even the network structure is unknown. The goal here is to discover the dependency relationships—the DAG itself—from the data. This is a more complex task known as structure learning or causal discovery. `pgmpy` provides several algorithms for this purpose [34, 44]:

* **Constraint-based Algorithms (e.g., `PC`):** These algorithms work by performing a series of statistical tests for conditional independence on the data to identify which variables are (and are not) directly connected.[41, 44]
* **Score-based Algorithms (e.g., `HillClimbSearch`):** These algorithms search through the space of possible DAGs, using a scoring function (like BIC or BDeu) to find the graph that best explains the data.[34]

This powerful capability allows Bayesian Networks to be used not just for encoding what is already known, but as a tool for exploratory data analysis and discovering new, potentially causal relationships within a complex system.

The journey from pure expert knowledge to purely data-driven discovery represents a continuum. A practitioner can start with a fully expert-defined model, use data to refine its parameters, or use data to discover the entire model from the ground up. The `pgmpy` library is expertly designed to support this entire workflow, making it an indispensable tool for anyone looking to apply probabilistic reasoning to complex problems.

## Conclusion

This tutorial has navigated the landscape of Bayesian statistics, from its core philosophical underpinnings to the practical construction of sophisticated models in Python. The journey began with the fundamental distinction between Bayesian and Frequentist thinking, establishing the Bayesian view of probability as a degree of belief that can be rationally updated via Bayes' Theorem. This principle of belief-updating is the engine that drives the entire framework.

It was then shown how Bayesian Networks scale this principle, using Directed Acyclic Graphs to represent the complex web of dependencies in a system. The DAG structure is not merely a visualization; it is a powerful assertion of conditional independencies that allows a massive joint probability distribution to be factorized into a set of small, manageable Conditional Probability Tables. This factorization is what makes modeling and inference computationally tractable.

The practical implementation using `pgmpy` demonstrated how these abstract concepts translate into concrete code. By defining a network structure, meticulously constructing `TabularCPD` objects, and associating them with the model, a complete probabilistic representation of a system can be built. The final step, inference, unlocks the model's predictive power, allowing for the calculation of posterior probabilities in light of new evidence.

Ultimately, Bayesian Networks offer a uniquely powerful synthesis of domain knowledge and data-driven learning. They provide a transparent, interpretable, and extensible framework for reasoning under uncertainty. Whether building a model from expert opinion, learning its parameters from data, or discovering its very structure, the principles of Bayesian inference and the tools provided by libraries like `pgmpy` equip the modern developer and data scientist with a robust methodology for tackling the ambiguity inherent in the real world.